In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading
import time
from tetris import TetrisGame

# --- ゲーム設定 ---
GAME_SPEED = 0.5  # ブロックが1段落ちるまでの秒数
BLOCK_SYMBOLS = {
    0: '⬜',
    1: '🟦',
    2: '🟨',
    3: '🟪',
    4: '🟧',
    5: '🟦',
    6: '🟩',
    7: '🟥',
}
BOARD_WIDTH = 10
BOARD_HEIGHT = 20

# --- グローバル変数 ---
game = TetrisGame(BOARD_WIDTH, BOARD_HEIGHT)
game_thread = None
stop_game_flag = False

# --- UIウィジェットの作成 ---
game_output = widgets.Output()
score_label = widgets.Label(value=f"Score: {game.score}")
game_over_label = widgets.Label(value="")

left_button = widgets.Button(description='◀️ Left')
right_button = widgets.Button(description='Right ▶️')
rotate_button = widgets.Button(description='Rotate 🔄')
drop_button = widgets.Button(description='Drop ⏬')
start_button = widgets.Button(description='Start Game', button_style='success')

controls = widgets.HBox([left_button, right_button, rotate_button, drop_button])
ui = widgets.VBox([score_label, game_over_label, game_output, controls, start_button])

# --- 描画関数 ---
def draw_game_state():
    with game_output:
        clear_output(wait=True)
        board_state = game.get_board_state()
        # HTMLとCSSを使って固定幅フォントで表示
        board_html = "<pre style='font-family: monospace; line-height: 1;'>"
        for row in board_state:
            board_html += ''.join([BLOCK_SYMBOLS.get(cell, ' ') for cell in row]) + '\n'
        board_html += "</pre>"
        display(widgets.HTML(board_html))
        score_label.value = f"Score: {game.score}"
        if game.game_over:
            game_over_label.value = "GAME OVER"

# --- ゲームループ ---
def game_loop():
    global stop_game_flag
    while not game.game_over and not stop_game_flag:
        game.step()
        draw_game_state()
        time.sleep(GAME_SPEED)
    
    # ゲームオーバー処理
    if game.game_over:
        start_button.description = 'Restart'
        start_button.disabled = False
        start_button.button_style = 'warning'

# --- ボタンのコールバック関数 ---
def on_left_button_clicked(b):
    if not game.game_over:
        game.move(-1)
        draw_game_state()

def on_right_button_clicked(b):
    if not game.game_over:
        game.move(1)
        draw_game_state()

def on_rotate_button_clicked(b):
    if not game.game_over:
        game.rotate()
        draw_game_state()

def on_drop_button_clicked(b):
    if not game.game_over:
        game.drop()
        draw_game_state()

def on_start_button_clicked(b):
    global game, game_thread, stop_game_flag
    
    # 既存のゲームスレッドを停止
    if game_thread and game_thread.is_alive():
        stop_game_flag = True
        game_thread.join()
        
    # ゲームをリセット
    game = TetrisGame(BOARD_WIDTH, BOARD_HEIGHT)
    stop_game_flag = False
    score_label.value = f"Score: {game.score}"
    game_over_label.value = ""
    start_button.disabled = True
    
    # 新しいゲームスreadを開始
    game_thread = threading.Thread(target=game_loop)
    game_thread.start()
    draw_game_state()

# --- イベントハンドラを登録 ---
left_button.on_click(on_left_button_clicked)
right_button.on_click(on_right_button_clicked)
rotate_button.on_click(on_rotate_button_clicked)
drop_button.on_click(on_drop_button_clicked)
start_button.on_click(on_start_button_clicked)

# --- アプリケーションの表示 ---
display(ui)
draw_game_state() # 初期盤面を描画